# ЛР3: SUPIR с маской на каждом шаге EDM
Включите **Internet** и **GPU** в настройках и подключите Dataset с `LW3_2_kaggle.zip`.
Запускайте сверху вниз. Сначала короткая серия; полная серия по умолчанию выключена.


In [ ]:
from pathlib import Path
from zipfile import ZipFile
import shutil
import subprocess
import sys
import os

WORK = Path('/kaggle/working')
PROJECT = WORK / 'LW3_2'
if not PROJECT.exists():
    candidates = list(Path('/kaggle/input').rglob('masked_supir.py'))
    if candidates:
        shutil.copytree(candidates[0].parent, PROJECT)
    else:
        archives = list(Path('/kaggle/input').rglob('LW3_2_kaggle.zip'))
        if not archives:
            raise FileNotFoundError('Подключите Dataset с архивом LW3_2_kaggle.zip через Add Input')
        with ZipFile(archives[0]) as z:
            for item in z.infolist():
                target = (WORK / item.filename).resolve()
                if not target.is_relative_to(WORK.resolve()):
                    raise ValueError('Недопустимый путь в архиве')
            z.extractall(WORK)
assert (PROJECT / 'experiment.py').exists()
ROOT = WORK / 'supir_lab'
os.environ['HF_HOME'] = str(ROOT / 'hf_cache')
os.environ['TORCH_HOME'] = str(ROOT / 'torch_cache')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
subprocess.run(['nvidia-smi'], check=True)
print('Проект:', PROJECT)


## Установка отдельной среды
Может занять несколько минут. Команда останавливается при конфликте зависимостей, не меняя системный PyTorch Kaggle.

In [ ]:
subprocess.run([sys.executable, str(PROJECT / 'setup_kaggle.py'), '--root', str(ROOT)], check=True)
PYTHON = ROOT / 'venv/bin/python'
def run(script, *args):
    subprocess.run([str(PYTHON), str(PROJECT / script), *map(str, args)], cwd=PROJECT, check=True)
subprocess.run([str(PYTHON), '-c',
    'import torch, xformers, pyiqa; print(torch.__version__); print(torch.cuda.get_device_name(0)); assert torch.cuda.is_available()'], check=True)


## Загрузка весов
По умолчанию SUPIR-v0F и SDXL. SUPIR скачивается из стороннего зеркала ashleykleynhans; ссылки и лицензии перечислены в README. LLaVA не нужна.

In [ ]:
run('download_weights.py', '--root', ROOT, '--variant', 'F')


## Проверки и подготовка входов
Без вызова модели: маски, деградации, план. Восстановление использует ту же папку с теми же параметрами.

In [ ]:
subprocess.run([str(PYTHON), '-m', 'unittest', 'discover', '-s', str(PROJECT), '-p', 'test_contracts.py', '-v'], cwd=PROJECT, check=True)
SHORT = WORK / 'results_supir_short'
run('experiment.py', '--input', PROJECT / 'input', '--output', SHORT, '--root', ROOT, '--prepare-only')


## Короткое исследование
Gaussian σ=15, три фотографии, baseline/zero/one/mixed. Метрики запускаются после завершения процесса SUPIR. При OOM не запускайте полный протокол; сохраните сообщение ошибки и проверьте доступную GPU/RAM.

In [ ]:
run('experiment.py', '--input', PROJECT / 'input', '--output', SHORT, '--root', ROOT)
run('evaluate.py', '--output', SHORT, '--device', 'cpu')


In [ ]:
from IPython.display import display, Markdown, Image
for path in sorted(SHORT.glob('comparison_*.png')):
    display(Image(filename=str(path)))
display(Markdown((SHORT / 'REPORT.md').read_text(encoding='utf-8').split('| Фото')[0]))


## Полный протокол (включите после короткого запуска)
Девять деградаций, формы маски, повтор seed и сила условия. Сохраняется отдельно; требуется существенно больше времени.

In [ ]:
RUN_FULL = False  # Измените на True после успешной короткой серии.
FULL = WORK / 'results_supir_full'
if RUN_FULL:
    run('experiment.py', '--input', PROJECT / 'input', '--output', FULL, '--root', ROOT, '--full')
    run('evaluate.py', '--output', FULL, '--device', 'cpu')


## Скачать результаты
Архив содержит измерения и сведения об окружении, без многогигабайтных весов. Скачайте ZIP из файлов `/kaggle/working`.

In [ ]:
from zipfile import ZIP_DEFLATED
archive_path = WORK / 'LW3_2_results.zip'
with ZipFile(archive_path, 'w', ZIP_DEFLATED) as archive:
    for folder in (SHORT, WORK / 'results_supir_full'):
        if folder.exists():
            for path in folder.rglob('*'):
                if path.is_file():
                    archive.write(path, path.relative_to(WORK))
    for name in ('setup.json', 'weights.json', 'environment.txt'):
        path = ROOT / name
        if path.exists():
            archive.write(path, 'environment/' + name)
print(archive_path)
from IPython.display import FileLink
display(FileLink(str(archive_path)))
